# Perfilado de datos: felicidad, 2015–2019

Objetivo: entender calidad, cobertura y diferencias de esquema antes de transformar.
Los CSV originales se conservan intactos. Los nombres comunes de este notebook son
una propuesta para comparar variables; no generan todavía un dataset transformado.

Ejecuta las celdas en orden. Los informes de YData son opcionales porque pueden tardar.
Instala dependencias desde la raíz: `.venv\Scripts\python.exe -m pip install -r requirements.txt`.

In [16]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

YEARS = range(2015, 2020)
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / 'data' / 'raw' / f'{y}.csv').is_file() for y in YEARS)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontraron los cinco CSV en data/raw.')
datasets = {y: pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / f'{y}.csv') for y in YEARS}
if any(df.empty for df in datasets.values()):
    raise ValueError('Hay un dataset vacío.')
display(pd.DataFrame([{'year': y, 'rows': len(df), 'columns': len(df.columns),
                       'duplicate_rows': int(df.duplicated().sum())}
                      for y, df in datasets.items()]))

,year,rows,columns,duplicate_rows
0,2015,158,12,0
1,2016,157,13,0
2,2017,155,12,0
3,2018,156,9,0
4,2019,156,9,0


## 1. Estructura y calidad por columna

Revisamos tipos observados, nulos, cardinalidad y cadenas vacías o con espacios.
No convertimos automáticamente los tipos: primero comprobamos si hay valores que fallan.

In [17]:
summaries = []
for year, df in datasets.items():
    summary = pd.DataFrame({'dtype': df.dtypes.astype(str),
                            'missing': df.isna().sum(),
                            'missing_pct': df.isna().mean().mul(100).round(2),
                            'distinct': df.nunique(dropna=True)})
    summary['blank_strings'] = [int(df[c].dropna().astype(str).str.strip().eq('').sum()) for c in df]
    summary['year'] = year
    summaries.append(summary.rename_axis('source_column').reset_index())
column_summary = pd.concat(summaries, ignore_index=True)
display(column_summary)
for year, df in datasets.items():
    print(f'{year}: primeras filas')
    display(df.head(3))
    if df.isna().any(axis=None):
        print('Registros con nulos:')
        display(df.loc[df.isna().any(axis=1)])

,source_column,dtype,missing,missing_pct,distinct,blank_strings,year
0,Country,object,0,0.00,158,0,2015
1,Region,object,0,0.00,10,0,2015
2,Happiness Rank,int64,0,0.00,157,0,2015
3,Happiness Score,float64,0,0.00,157,0,2015
4,Standard Error,float64,0,0.00,153,0,2015
5,Economy (GDP per Capita),float64,0,0.00,158,0,2015
6,Family,float64,0,0.00,158,0,2015
7,Health (Life Expectancy),float64,0,0.00,157,0,2015
8,Freedom,float64,0,0.00,158,0,2015
9,Trust (Government Corruption),float64,0,0.00,157,0,2015


2015: primeras filas


,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204


2016: primeras filas


,Country,Region,Happiness Rank,Happiness Score,Lower Confidence Interval,Upper Confidence Interval,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Denmark,Western Europe,1,7.526,7.460,7.592,1.44178,1.16374,0.79504,0.57941,0.44453,0.36171,2.73939
1,Switzerland,Western Europe,2,7.509,7.428,7.590,1.52733,1.14524,0.86303,0.58557,0.41203,0.28083,2.69463
2,Iceland,Western Europe,3,7.501,7.333,7.669,1.42666,1.18326,0.86733,0.56624,0.14975,0.47678,2.83137


2017: primeras filas


... (3 outputs left out)

Registros con nulos:


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
19,20,United Arab Emirates,6.774,2.096,0.776,0.67,0.284,0.186,NaN


2019: primeras filas


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341


## 2. Comparación semántica del esquema

Cambiar espacios por guiones bajos no basta: `Family` y `Social support`, por ejemplo,
necesitan una correspondencia explícita. Esta tabla propone equivalencias para revisión.
Que compartan nombre no garantiza que la metodología o escala sea idéntica entre años.

In [18]:
ALIASES = {
    'country': ['Country', 'Country or region'],
    'region': ['Region'],
    'happiness_rank': ['Happiness Rank', 'Happiness.Rank', 'Overall rank'],
    'happiness_score': ['Happiness Score', 'Happiness.Score', 'Score'],
    'gdp_per_capita': ['Economy (GDP per Capita)', 'Economy..GDP.per.Capita.', 'GDP per capita'],
    'social_support': ['Family', 'Social support'],
    'healthy_life_expectancy': ['Health (Life Expectancy)', 'Health..Life.Expectancy.', 'Healthy life expectancy'],
    'freedom': ['Freedom', 'Freedom to make life choices'],
    'corruption': ['Trust (Government Corruption)', 'Trust..Government.Corruption.', 'Perceptions of corruption'],
    'generosity': ['Generosity'],
    'dystopia_residual': ['Dystopia Residual', 'Dystopia.Residual'],
    'standard_error': ['Standard Error'],
    'lower_confidence_interval': ['Lower Confidence Interval'],
    'upper_confidence_interval': ['Upper Confidence Interval'],
    'whisker_low': ['Whisker.low'],
    'whisker_high': ['Whisker.high'],
}
source_columns = {}
for year, df in datasets.items():
    source_columns[year] = {canonical: next((c for c in aliases if c in df), None)
                            for canonical, aliases in ALIASES.items()}
    mapped = {c for c in source_columns[year].values() if c is not None}
    if mapped != set(df.columns):
        raise ValueError(f'Columnas sin correspondencia en {year}: {set(df.columns) - mapped}')
schema_comparison = pd.DataFrame(source_columns).rename_axis('proposed_name')
display(schema_comparison.fillna('—'))
common_columns = schema_comparison.index[schema_comparison.notna().all(axis=1)].tolist()
print('Variables comunes:', common_columns)

,2015,2016,2017,2018,2019
proposed_name,,,,,
country,Country,Country,Country,Country or region,Country or region
region,Region,Region,—,—,—
happiness_rank,Happiness Rank,Happiness Rank,Happiness.Rank,Overall rank,Overall rank
happiness_score,Happiness Score,Happiness Score,Happiness.Score,Score,Score
gdp_per_capita,Economy (GDP per Capita),Economy (GDP per Capita),Economy..GDP.per.Capita.,GDP per capita,GDP per capita
social_support,Family,Family,Family,Social support,Social support
healthy_life_expectancy,Health (Life Expectancy),Health (Life Expectancy),Health..Life.Expectancy.,Healthy life expectancy,Healthy life expectancy
freedom,Freedom,Freedom,Freedom,Freedom to make life choices,Freedom to make life choices
corruption,Trust (Government Corruption),Trust (Government Corruption),Trust..Government.Corruption.,Perceptions of corruption,Perceptions of corruption


Variables comunes: ['country', 'happiness_rank', 'happiness_score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'corruption', 'generosity']


## 3. Tipos numéricos, claves y categorías

La clave candidata es país–año. Los espacios y cambios de nombre de países requieren
revisión; aplicar mayúsculas/minúsculas no resuelve equivalencias geográficas.
Se inspeccionan diferencias entre años consecutivos sin inferir automáticamente un alias.

In [19]:
numeric_checks = []
country_sets = {}
for year, df in datasets.items():
    mapping = source_columns[year]
    country = df[mapping['country']]
    stripped = country.astype('string').str.strip()
    country_sets[year] = set(stripped.dropna())
    print(f'{year}: países repetidos={country.duplicated().sum()}, '
          f'espacios exteriores={(country.notna() & country.ne(stripped)).sum()}, '
          f'colisiones al ignorar caja/espacios={stripped.str.casefold().duplicated().sum()}')
    display(df.loc[country.duplicated(keep=False)])
    for canonical, source in mapping.items():
        if source is None or canonical in ('country', 'region'):
            continue
        converted = pd.to_numeric(df[source], errors='coerce')
        numeric_checks.append({'year': year, 'variable': canonical,
                               'source_dtype': str(df[source].dtype),
                               'conversion_failures': int((df[source].notna() & converted.isna()).sum())})
display(pd.DataFrame(numeric_checks))
display(pd.DataFrame([{'year': y, 'only_previous_year': sorted(country_sets[y-1] - country_sets[y]),
                       'only_current_year': sorted(country_sets[y] - country_sets[y-1])}
                      for y in YEARS if y > 2015]))

2015: países repetidos=0, espacios exteriores=0, colisiones al ignorar caja/espacios=0


,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual


2016: países repetidos=0, espacios exteriores=0, colisiones al ignorar caja/espacios=0


,Country,Region,Happiness Rank,Happiness Score,Lower Confidence Interval,Upper Confidence Interval,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual


2017: países repetidos=0, espacios exteriores=0, colisiones al ignorar caja/espacios=0


,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual


... (2 outputs left out)

2019: países repetidos=0, espacios exteriores=0, colisiones al ignorar caja/espacios=0


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption


,year,variable,source_dtype,conversion_failures
0,2015,happiness_rank,int64,0
1,2015,happiness_score,float64,0
2,2015,gdp_per_capita,float64,0
3,2015,social_support,float64,0
4,2015,healthy_life_expectancy,float64,0
5,2015,freedom,float64,0
6,2015,corruption,float64,0
7,2015,generosity,float64,0
8,2015,dystopia_residual,float64,0
9,2015,standard_error,float64,0


,year,only_previous_year,only_current_year
0,2016,"[Central African Republic, Djibouti, Lesotho, ...","[Belize, Namibia, Puerto Rico, Somalia, Somali..."
1,2017,"[Comoros, Hong Kong, Laos, Puerto Rico, Somali...","[Central African Republic, Hong Kong S.A.R., C..."
2,2018,"[Hong Kong S.A.R., China, North Cyprus, Taiwan...","[Hong Kong, Laos, Northern Cyprus, Taiwan, Tri..."
3,2019,"[Angola, Belize, Macedonia, Sudan]","[Comoros, Gambia, North Macedonia, Swaziland]"


## 4. Distribuciones y posibles atípicos

La regla de 1,5 × IQR señala observaciones para revisar; no justifica eliminarlas.
Los indicadores no deben interpretarse automáticamente como PIB monetario o años de vida.

In [20]:
outliers = []
for year, df in datasets.items():
    for canonical, source in source_columns[year].items():
        if source is None or canonical in ('country', 'region', 'happiness_rank'):
            continue
        values = pd.to_numeric(df[source], errors='coerce')
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        flagged = values.lt(q1 - 1.5 * iqr) | values.gt(q3 + 1.5 * iqr)
        outliers.append({'year': year, 'variable': canonical, 'min': values.min(),
                         'median': values.median(), 'max': values.max(),
                         'iqr_flags': int(flagged.sum()),
                         'countries': df.loc[flagged, source_columns[year]['country']].tolist()})
display(pd.DataFrame(outliers))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for year, df in datasets.items():
    score = df[source_columns[year]['happiness_score']].dropna()
    axes[0].hist(score, bins=range(0, 11), histtype='step', density=True, label=str(year))
axes[0].set(title='Distribución del puntaje por año', xlabel='Happiness score', ylabel='Densidad')
axes[0].legend(title='Año')
df = datasets[2015]
top = df.nlargest(15, source_columns[2015]['happiness_score']).sort_values(source_columns[2015]['happiness_score'])
axes[1].barh(top['Country'], top['Happiness Score'])
axes[1].set(title='15 países con mayor puntaje, 2015', xlabel='Happiness score')
fig.tight_layout()
output_dir = PROJECT_ROOT / 'dashboards' / 'screenshots'
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'eda_score_distribucion.png', bbox_inches='tight', dpi=150)
plt.show()

,year,variable,min,median,max,iqr_flags,countries
0,2015,happiness_score,2.839000,5.232500,7.587000,0,[]
1,2015,gdp_per_capita,0.000000,0.910245,1.690420,0,[]
2,2015,social_support,0.000000,1.029510,1.402230,3,"[Central African Republic, Afghanistan, Togo]"
3,2015,healthy_life_expectancy,0.000000,0.696705,1.025250,0,[]
4,2015,freedom,0.000000,0.435515,0.669730,0,[]
5,2015,corruption,0.000000,0.107220,0.551910,14,"[Switzerland, Denmark, Norway, Finland, Sweden..."
6,2015,generosity,0.000000,0.216130,0.795880,2,"[Thailand, Myanmar]"
7,2015,dystopia_residual,0.328580,2.095415,3.602140,5,"[Mexico, Hong Kong, Sri Lanka, Rwanda, Syria]"
8,2015,standard_error,0.018480,0.043940,0.136930,10,"[Suriname, Trinidad and Tobago, Libya, Jamaica..."
9,2016,happiness_score,2.905000,5.314000,7.526000,0,[]


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11564\3735720008.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Informes exploratorios opcionales

Activa la bandera para generar un informe por año. El diagnóstico anterior funciona sin
ejecutar los informes. Los HTML se guardan en `data/reports` para consultarlos aparte.

In [22]:
GENERATE_REPORTS = True
if GENERATE_REPORTS:
    from ydata_profiling import ProfileReport
    report_dir = PROJECT_ROOT / 'data' / 'reports'
    report_dir.mkdir(parents=True, exist_ok=True)
    for year, df in datasets.items():
        report = ProfileReport(df, title=f'Perfil de datos — {year}', explorative=True)
        report.to_file(report_dir / f'profile_{year}.html')
else:
    print('Informes opcionales desactivados. Cambia GENERATE_REPORTS a True para generarlos.')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 76.07it/s]


## 6. Hallazgos y decisiones para conversar

Los archivos actuales tienen **782 filas**: 158, 157, 155, 156 y 156 por año.
No hay filas completamente duplicadas. Existe un nulo en `Perceptions of corruption`
de **United Arab Emirates, 2018**. Los tipos numéricos se verifican arriba;
no es necesario convertir una columna que ya tiene el tipo correcto.

### Diferencias de estructura

| Variable adicional | Disponibilidad |
| --- | --- |
| Región | 2015–2016 |
| Error estándar | 2015 |
| Intervalos inferior/superior de confianza | 2016 |
| Whisker inferior/superior | 2017 |
| Residuo de distopía | 2015–2017 |

Corrección del hallazgo original: 2017 no tiene región ni las columnas de intervalos
de 2016; tiene `Whisker.low` y `Whisker.high`. No equiparamos estas medidas sin revisar
la documentación de origen.

### Propuestas pendientes de tu decisión

1. **Esquema:** conservar todas las columnas en una tabla histórica con `year` y nulos
   estructurales, y preparar después una vista de modelado con las nueve variables comunes.
   Así conservamos información sin imputar campos que nunca fueron publicados en ciertos años.
2. **Nombres:** usar los nombres `snake_case` de la tabla de correspondencias. Revisar
   equivalencia metodológica de los indicadores antes de interpretar tendencias.
3. **Categorías:** quitar espacios exteriores y revisar espacios repetidos. Tu propuesta
   de primera letra mayúscula y resto minúscula convertiría `United States` en
   `United states`; propongo conservar nombres de presentación y usar una clave separada
   para comparar. Revisar manualmente alias de países antes de unirlos.
4. **Nulo de 2018:** considerar mediana para corrupción, validando la distribución y
   el efecto de la imputación. Para ML, ajustar el imputador solo en entrenamiento.
   La existencia de un nulo por sí sola no demuestra que la mediana sea la mejor opción.
5. **Atípicos y duplicados:** revisar los señalados antes de borrar; países con valores
   extremos pueden ser válidos. Comprobar unicidad de país–año después de resolver alias.
6. **Modelo futuro:** `happiness_score` sería el objetivo. Excluir `happiness_rank` de
   los predictores porque deriva del puntaje; revisar también residuo e intervalos por
   su relación con el objetivo. Decidir partición temporal antes de imputar o escalar.
   No aplicar escalado todavía: depende del modelo y se ajusta solo en entrenamiento.

**Siguiente conversación:** escoger el esquema histórico y la política de nombres de
países. Después implementaremos las decisiones acordadas en `transform_data.ipynb`.
Este notebook no modifica los CSV ni implementa la transformación.